# Stage 3 — QLoRA Fine-tuning (Qwen3-1.7B, Spider) via Unsloth

Orchestrator notebook only — real logic lives in `src/train_qlora.py`. Reuses the same Drive project folder as the Stage 1 baseline notebook.

## Resumability -- read this first

`--output_dir` is a path **inside your Drive project folder**, so checkpoints (adapter weights + optimizer/scheduler/RNG state) survive a Colab disconnect. `src/train_qlora.py` automatically resumes from the latest checkpoint found there.

**If Colab disconnects mid-training**: just reopen this notebook, run the setup cells again (mount Drive, cd, install deps), then re-run the training cell **unchanged** -- it detects the existing checkpoint and continues instead of starting over. You lose at most `--save_steps` steps of progress (default 50).

Runtime: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
!nvidia-smi

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

PROJECT_DIR = '/content/drive/MyDrive/finetuning_project'

required = ['src/train_qlora.py', 'src/generate_predictions.py', 'requirements-train.txt']
missing = [f for f in required if not os.path.exists(os.path.join(PROJECT_DIR, f))]
if missing:
    raise FileNotFoundError(f"Missing {missing} under {PROJECT_DIR}. Upload the latest project folder first.")
print('Project files found at', PROJECT_DIR)

Project files found at /content/drive/MyDrive/finetuning_project


In [5]:
%cd {PROJECT_DIR}

/content/drive/.shortcut-targets-by-id/1Ufrz4U98lHEBosFzZKgpz1newqoo9lis/finetuning_project


In [6]:
!pip install -q -r requirements-train.txt
# If this errors on a specific trl/unsloth argument later (API churn between
# versions), check the installed trl version's SFTTrainer/SFTConfig signature.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 836.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 132.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [7]:
# Reuses the Spider data already downloaded for Stage 1 -- just checking the
# training-specific files are also there.
train_required = ['data/spider/train_spider.json', 'data/spider/train_others.json', 'data/spider/tables.json']
missing_data = [f for f in train_required if not os.path.exists(f)]
if missing_data:
    raise FileNotFoundError(
        f"Missing {missing_data}. Run the Stage 1 notebook's download cell first "
        "(same data/spider/ folder is reused here)."
    )
print('Spider training data found.')

Spider training data found.


## Smoke test (tiny subset, 1 epoch)

Run this first to confirm the training loop actually works -- model loads, LoRA applies, a checkpoint saves -- before committing GPU hours to the full dataset.

In [ ]:
!python src/train_qlora.py --model_name unsloth/Qwen3-1.7B --data_dir data/spider --output_dir results/qlora_qwen3_1p7b_smoke --limit 20 --num_train_epochs 1 --save_steps 5

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
model.safetensors: 100% 1.41G/1.41G [00:10<00:00, 134MB/s]
Loading weights: 100% 310/310 [00:01<00:00, 240.35it/s]
generation_config.json: 100% 237/237 [00:00<00:00, 1.50MB/s]
config.json: 100% 1.36k/1.36k [00:00<00:00, 4.43MB/s]
tokenizer_config.json: 100% 10.5k/10.5k [00:00<00:00, 19.5MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 112MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 102MB/s]
tokenizer.json

## Full fine-tuning run

Only run this once the smoke test above completes cleanly.

**After a disconnect, just re-run this exact cell** -- `--output_dir` already has checkpoints in it, so training resumes instead of restarting.

In [10]:
!python src/train_qlora.py --model_name unsloth/Qwen3-1.7B --data_dir data/spider --output_dir results/qlora_qwen3_1p7b --num_train_epochs 3 --save_steps 50 --save_total_limit 3

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Found completed adapter at results/qlora_qwen3_1p7b/final_adapter -- skipping training, retrying merge only.
==((====))==  Unsloth 2026.6.9: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100% 310/310 [00:02<00:00, 124.72it/s]
unsloth/qwen3-1.7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Unsloth 2026.6.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
config.json: 100% 752/752 [00:00<00:00, 1.92MB/s]
Unsloth:

## Evaluate the fine-tuned model

Reuses the exact same Stage 1 scripts, pointed at the merged fine-tuned model instead of the base model -- so these results are directly comparable to `results/baseline_qwen3_1p7b/`.

In [11]:
!python src/generate_predictions.py --model_name results/qlora_qwen3_1p7b/merged_model --data_dir data/spider --output_dir results/finetuned_qwen3_1p7b --dtype float16

The tokenizer you are loading from 'results/qlora_qwen3_1p7b/merged_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100% 310/310 [00:10<00:00, 30.60it/s]
Generating:   0% 0/1034 [00:00<?, ?it/s]Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Generating:   0% 1/1034 [00:01<26:49,  1.56s/it]Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please r

In [12]:
!python src/run_eval.py --gold results/finetuned_qwen3_1p7b/gold.txt --pred results/finetuned_qwen3_1p7b/pred.txt --db data/spider/database --table data/spider/tables.json --output_dir results/finetuned_qwen3_1p7b

medium pred: SELECT song_name ,  song_release_year FROM singer WHERE age  =  (SELECT min(age) FROM singer)
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

medium pred: SELECT max(Height) ,  avg(Height) FROM stadium
medium gold: select max(capacity), average from stadium

medium pred: SELECT avg(Capacity) ,  max(Highest) FROM stadium
medium gold: select avg(capacity) ,  max(capacity) from stadium

medium pred: SELECT T3.Stadium_ID ,  count(*) FROM concert AS T1 JOIN stadium AS T3 ON T1.Stadium_ID  =  T3.Stadium_ID GROUP BY T3.Stadium_ID
medium gold: SELECT T2.name ,  count(*) FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id GROUP BY T1.stadium_id

extra pred: SELECT T3.Name ,  T3.Capacity FROM concert AS T1 JOIN stadium AS T2 ON T1.Stadium_ID  =  T2.Stadium_ID JOIN singer_in_concert AS T4 ON T1.concert_ID  =  T4.concert_ID WHERE T1.Year  >=  2014 GROUP BY T3.Stadium_ID ORDER BY count(*) DESC LIMIT 1
extra gold: SELECT T2.name 

In [13]:
!python src/error_analysis.py --debug_file results/finetuned_qwen3_1p7b/predictions_debug.jsonl --output_dir results/finetuned_qwen3_1p7b

{
  "total_examples": 1034,
  "buckets": {
    "exact_match": {
      "count": 416,
      "pct": 40.2
    },
    "other_non_exact_match": {
      "count": 328,
      "pct": 31.7
    },
    "hallucinated_table": {
      "count": 139,
      "pct": 13.4
    },
    "missing_table": {
      "count": 71,
      "pct": 6.9
    },
    "missing_join": {
      "count": 30,
      "pct": 2.9
    },
    "aggregation_mismatch": {
      "count": 18,
      "pct": 1.7
    },
    "missing_set_operation": {
      "count": 13,
      "pct": 1.3
    },
    "missing_order_by": {
      "count": 8,
      "pct": 0.8
    },
    "missing_group_by": {
      "count": 6,
      "pct": 0.6
    },
    "subquery_vs_join_style": {
      "count": 5,
      "pct": 0.5
    }
  }
}

Sample report saved to results/finetuned_qwen3_1p7b/error_analysis_samples.md


## Compare against baseline

Now `results/baseline_qwen3_1p7b/` and `results/finetuned_qwen3_1p7b/` each have `eval_log.txt` (EM/EX by difficulty), `perf_metrics.json` (latency/tokens-per-sec/GPU memory), and `error_analysis.json` (failure-mode buckets) -- diff them directly for the before/after story.